# Customer Segmentation Label Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `segment`

## 2 · Project Overview

We predict **customer segments** (At-Risk, Casual, Loyal, VIP) based on RFM metrics (recency, frequency, monetary), tenure, and shopping channel.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a customer's recency, purchase frequency, monetary value, average order value, tenure, and shopping channel, classify them into one of 4 segments.

## 5 · Why This Project Matters

- **Customer segmentation** drives targeted marketing strategies.
- Predicting segment labels enables real-time personalization.
- Understanding segment drivers helps optimize retention efforts.
- RFM analysis is a foundational CRM technique.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | recency_days, frequency, monetary, avg_order_value, tenure_months, channel |
| **Target** | segment (4 classes: At-Risk, Casual, Loyal, VIP) |
| **Class balance** | ~25% each |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "segment"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: segment
Save dir: E:\Github\Machine-Learning-Projects\Classification\Customer Segmentation Label Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 customers with purchasing behavior and 4 segment labels.

In [4]:
np.random.seed(SEED)
n = 6000
recency_days = np.random.poisson(30, n).clip(1, 365)
frequency = np.random.poisson(8, n).clip(1, 100)
monetary = np.round(np.random.lognormal(5, 1, n).clip(10, 10000), 2)
avg_order_value = np.round(monetary / (frequency + 1), 2)
tenure_months = np.random.randint(1, 72, n)
channel = np.random.choice(["Online", "In-Store", "Both"], n, p=[0.4, 0.3, 0.3])
chan_num = np.where(channel == "Online", 0, np.where(channel == "In-Store", 1, 2))

score = (-0.005 * recency_days + 0.05 * frequency + 0.0003 * monetary
         + 0.01 * tenure_months + 0.1 * chan_num + np.random.normal(0, 0.8, n))
segment = np.where(score < np.percentile(score, 25), "At-Risk",
          np.where(score < np.percentile(score, 50), "Casual",
          np.where(score < np.percentile(score, 75), "Loyal", "VIP")))

df = pd.DataFrame({"recency_days": recency_days, "frequency": frequency,
                    "monetary": monetary, "avg_order_value": avg_order_value,
                    "tenure_months": tenure_months, "channel": channel,
                    "segment": segment})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['segment'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (6000, 7)
Class balance:
segment
Casual     0.25
VIP        0.25
At-Risk    0.25
Loyal      0.25
Name: proportion, dtype: float64


,recency_days,frequency,monetary,avg_order_value,tenure_months,channel,segment
0,28,9,10.70,1.07,32,Online,Casual
1,34,9,558.19,55.82,28,Both,Casual
2,23,9,90.59,9.06,29,In-Store,Casual
3,32,7,339.10,42.39,49,In-Store,VIP
4,36,13,36.90,2.64,36,In-Store,At-Risk


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (6000, 7)

Missing values:
recency_days       0
frequency          0
monetary           0
avg_order_value    0
tenure_months      0
channel            0
segment            0
dtype: int64

Duplicate rows: 0

Dtypes:
recency_days         int32
frequency            int32
monetary           float64
avg_order_value    float64
tenure_months        int32
channel             object
segment             object
dtype: object

Target 'segment' confirmed. Value counts:
segment
Casual     1500
VIP        1500
At-Risk    1500
Loyal      1500
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["recency_days", "frequency", "monetary", "avg_order_value", "tenure_months"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Segment", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `segment`.

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ["At-Risk", "Casual", "Loyal", "VIP"]
df[TARGET].value_counts().reindex(order).plot(kind="bar", ax=ax,
    color=["salmon", "orange", "steelblue", "gold"], edgecolor="black")
ax.set_title("Customer Segment Distribution")
plt.tight_layout()
plt.show()
print(f"Segment counts:\n{df[TARGET].value_counts()}")

Segment counts:
segment
Casual     1500
VIP        1500
At-Risk    1500
Loyal      1500
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4800, 6), Test: (1200, 6)
Train class distribution:
segment
0    0.25
2    0.25
3    0.25
1    0.25
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `channel` encoded with OrdinalEncoder. Target encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~25% per segment (by design).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["rfm_score"] = (X_train["frequency"] * X_train["monetary"]) / (X_train["recency_days"] + 1)
X_test["rfm_score"] = (X_test["frequency"] * X_test["monetary"]) / (X_test["recency_days"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (7): ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'tenure_months', 'channel', 'rfm_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.3358
F1       : 0.2876

              precision    recall  f1-score   support

           0       0.37      0.61      0.46       300
           1       0.16      0.06      0.09       300
           2       0.25      0.11      0.15       300
           3       0.37      0.56      0.44       300

    accuracy                           0.34      1200
   macro avg       0.29      0.34      0.29      1200
weighted avg       0.29      0.34      0.29      1200



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
SVC                            0.354167           0.354167       NaN  0.323993   0.337738  0.354167    1.210736
LogisticRegression             0.350000           0.350000  0.610840  0.305465   0.311923  0.350000    0.030252
CalibratedClassifierCV         0.348333           0.348333  0.608353  0.288385   0.301925  0.348333    0.091518
RidgeClassifierCV              0.346667           0.346667       NaN  0.262280   0.295134  0.346667    0.016333
LinearDiscriminantAnalysis     0.346667           0.346667  0.609895  0.301609   0.304794  0.346667    0.017802
LinearSVC                      0.344167           0.344167       NaN  0.259655   0.288160  0.344167    0.023986
RidgeClassifier                0.344167           0.344167       NaN  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: rf
Accuracy: 0.3225
F1: 0.3068


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.3180  (2.7s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[31]	valid_0's multi_logloss: 1.33676
LightGBM F1: 0.3243  (1.6s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               0.3433  0.3243     0.3258  0.3433
CatBoost               0.3483  0.3180     0.3238  0.3483
FLAML                  0.3225  0.3068     0.3071  0.3225
Logistic Regression    0.3358  0.2876     0.2872  0.3358

Top 3 models: ['LightGBM', 'CatBoost', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  LightGBM:
    Accuracy : 0.3433
    F1       : 0.3243
    Precision: 0.3258
    Recall   : 0.3433

  CatBoost:
    Accuracy : 0.3483
    F1       : 0.3180
    Precision: 0.3238
    Recall   : 0.3483

  FLAML:
    Accuracy : 0.3225
    F1       : 0.3068
    Precision: 0.3071
    Recall   : 0.3225


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: LightGBM

Classification Report:
              precision    recall  f1-score   support

           0       0.38      0.53      0.44       300
           1       0.25      0.15      0.19       300
           2       0.31      0.21      0.25       300
           3       0.37      0.49      0.42       300

    accuracy                           0.34      1200
   macro avg       0.33      0.34      0.32      1200
weighted avg       0.33      0.34      0.32      1200


Total errors: 788 / 1200 (65.7%)

Sample misclassifications:
      recency_days  frequency  monetary  avg_order_value  tenure_months  channel   rfm_score  actual  predicted  correct
172             33          8   1086.24           120.69             59      2.0  255.585882       2          3    False
3614            22         11     60.34             5.03             22      0.0   28.858261       2          3    False
2898            37         10    222.70            20.25             71      0.0   58.605263   

## 25 · Interpretation and Business Insight

**Key findings:**
- **Frequency** and **monetary** are the strongest segment discriminators.
- **Recency** separates At-Risk from active segments.
- **RFM score** (engineered) is highly predictive.
- **Channel** adds marginal but useful signal.

**Business takeaway:** Focus retention campaigns on At-Risk customers with declining frequency and rising recency.

## 26 · Limitations

1. Synthetic segments — real segmentation uses unsupervised methods.
2. No product category preferences.
3. Missing customer satisfaction data.
4. No seasonal purchasing patterns.
5. Equal segment sizes are unrealistic.

## 27 · How to Improve This Project

1. Use real CRM/transaction data.
2. Derive segments from clustering (UMAP + HDBSCAN).
3. Add product affinity features.
4. Include customer satisfaction scores.
5. Model segment transitions over time.

## 28 · Production Considerations

- Deploy for real-time customer scoring.
- Integrate with marketing automation.
- Monitor segment drift monthly.
- Output segment probabilities for soft assignment.
- Update segmentation quarterly.

## 29 · Common Mistakes

1. Treating segments as fixed labels (they evolve).
2. Not considering segment size imbalance in real data.
3. Using accuracy without per-segment analysis.
4. Ignoring the ordinal nature of segments.
5. Not validating against business KPIs.

## 30 · Mini Challenge / Exercises

1. Remove `monetary` — how much does accuracy drop?
2. Try ordinal classification (ordered segments).
3. Compare 4-segment vs. 3-segment (merge Casual + Loyal).
4. Plot decision boundaries for top 2 features.
5. Analyze segment transition patterns over tenure.

## 31 · Final Summary / Key Takeaways

1. **RFM metrics** are the foundation of customer segmentation.
2. **Frequency and monetary** are the strongest discriminators.
3. **Recency** identifies at-risk customers.
4. **Engineered RFM score** combines signals effectively.
5. **Real segmentation** typically uses unsupervised methods first.
6. **Segment prediction** enables real-time personalization.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Customer Segmentation Label Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.3483,
    "f1": 0.318,
    "precision": 0.3238,
    "recall": 0.3483
  },
  "LightGBM": {
    "accuracy": 0.3433,
    "f1": 0.3243,
    "precision": 0.3258,
    "recall": 0.3433
  },
  "Logistic Regression": {
    "accuracy": 0.3358,
    "f1": 0.2876,
    "precision": 0.2872,
    "recall": 0.3358
  },
  "FLAML": {
    "accuracy": 0.3225,
    "f1": 0.3068,
    "precision": 0.3071,
    "recall": 0.3225
  }
}
